In [3]:
import boto3
import base64
import time

ec2 = boto3.client('ec2')
asg_client = boto3.client('autoscaling')
cloudwatch_client = boto3.client('cloudwatch')

ami_id = 'ami-0522ab6e1ddcc7055'  
instance_type = 't2.micro'
key_name = 'Key_Pair'  

with open('startup.sh', 'r') as file:
    user_data_script = file.read()

launch_configuration = asg_client.create_launch_configuration(
    LaunchConfigurationName='web-server-lc',
    ImageId=ami_id, 
    InstanceType=instance_type,
    SecurityGroups=['sg-0006b3cf7416c4a30'],
    UserData=user_data_script,
)

auto_scaling_group = asg_client.create_auto_scaling_group(
    AutoScalingGroupName='web-asg',
    LaunchConfigurationName='web-server-lc',
    MinSize=1,
    MaxSize=3,
    DesiredCapacity=1,
    AvailabilityZones=['ap-south-1a', 'ap-south-1b'],  
)

scale_up_policy = asg_client.put_scaling_policy(
    AutoScalingGroupName='web-asg',
    PolicyName='scale-up',
    AdjustmentType='ChangeInCapacity',
    ScalingAdjustment=1,
    Cooldown=300,
)

scale_down_policy = asg_client.put_scaling_policy(
    AutoScalingGroupName='web-asg',
    PolicyName='scale-down',
    AdjustmentType='ChangeInCapacity',
    ScalingAdjustment=-1,
    Cooldown=300,
)

cloudwatch_client.put_metric_alarm(
    AlarmName='HighCPUAlarm',
    MetricName='CPUUtilization',
    Namespace='AWS/EC2',
    Statistic='Average',
    Period=300,
    EvaluationPeriods=1,
    Threshold=60.0,
    ComparisonOperator='GreaterThanThreshold',
    Dimensions=[
        {
            'Name': 'AutoScalingGroupName',
            'Value': 'web-asg'
        },
    ],
    AlarmActions=[scale_up_policy['PolicyARN']],
)

cloudwatch_client.put_metric_alarm(
    AlarmName='LowCPUAlarm',
    MetricName='CPUUtilization',
    Namespace='AWS/EC2',
    Statistic='Average',
    Period=300,
    EvaluationPeriods=1,
    Threshold=30.0,
    ComparisonOperator='LessThanThreshold',
    Dimensions=[
        {
            'Name': 'AutoScalingGroupName',
            'Value': 'web-asg'
        },
    ],
    AlarmActions=[scale_down_policy['PolicyARN']],
)

print("Auto Scaling group and policies created successfully.")
time.sleep(60)
asg_response = asg_client.describe_auto_scaling_groups(
    AutoScalingGroupNames=['web-asg']
)

instance_ids = []
for group in asg_response['AutoScalingGroups']:
    instance_ids.extend([instance['InstanceId'] for instance in group['Instances']])

print(f'Instances in Auto Scaling group: {instance_ids}')

if instance_ids:
    instance_description = ec2.describe_instances(InstanceIds=instance_ids)
    for reservation in instance_description['Reservations']:
        for instance in reservation['Instances']:
            instance_id = instance['InstanceId']
            public_ip = instance.get('PublicIpAddress', 'No Public IP assigned')
            print(f'Instance ID: {instance_id}, Public IP: {public_ip}')
else:
    print("No instances found in the Auto Scaling group.")


Auto Scaling group and policies created successfully.
Instances in Auto Scaling group: ['i-0e8bc23f844442daf']
Instance ID: i-0e8bc23f844442daf, Public IP: 13.235.94.26
